In [1]:
include("../RayTracing.jl")

Main.RayTracing

In [2]:
triangles = RayTracing.parse_obj(
    "/home/jmyslinski/random_stuff/spherical-cow/examples/objects/cow.obj",
    RayTracing.Translate(RayTracing.Pnt3(0, 0, 0)),
    false,
    false,
    nothing
)

┌ Warning: Skipping something: s 1
└ @ Main.RayTracing /home/jmyslinski/random_stuff/PBRJ/src/parsers/parse_obj.jl:222


1-element Vector{Any}:
 Main.RayTracing.Triangle[Main.RayTracing.Triangle(Main.RayTracing.ShapeCore(Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), false, false), Main.RayTracing.Pnt3[[2.292449, -0.871852, -0.8824], [2.407309, -0.97498, -0.805091], [2.229345, -0.992723, -0.862826]], nothing, nothing, nothing, nothing), Main.RayTracing.Triangle(Main.RayTracing.ShapeCore(Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0

In [ ]:
function voxelize_to_bounds(triangles::Vector{RayTracing.Triangle}, voxel_size::Float64; samples_per_edge=20)
    voxel_bounds = RayTracing.Bounds3[]
    seen_voxels = Set{RayTracing.Pnt3}()
    
    step = 1.0 / samples_per_edge
    
    for triangle in triangles
        v0, v1, v2 = triangle.vertices[1], triangle.vertices[2], triangle.vertices[3]
        
        # Sample triangle more densely based on its size
        edge_lengths = [
            RayTracing.norm(v1 - v0),
            RayTracing.norm(v2 - v1), 
            RayTracing.norm(v0 - v2)
        ]
        max_edge = maximum(edge_lengths)
        
        # More samples for larger triangles
        local_step = min(step, voxel_size / max_edge)
        
        u = 0.0
        while u <= 1.0
            v = 0.0
            while v <= (1.0 - u)
                w = 1.0 - u - v
                point = w*v0 + u*v1 + v*v2
                
                voxel_coord = floor.(Int, point / voxel_size)
                
                if voxel_coord ∉ seen_voxels
                    push!(seen_voxels, voxel_coord)
                    
                    pMin = voxel_coord .* voxel_size
                    pMax = pMin .+ voxel_size
                    push!(voxel_bounds, RayTracing.Bounds3(pMin, pMax))
                end
                
                v += local_step
            end
            u += local_step
        end
    end
    
    return voxel_bounds
end

voxelize_to_bounds (generic function with 1 method)

In [5]:
voxelize_to_bounds(triangles[1], 1.0)


MethodError: MethodError: Cannot `convert` an object of type StaticArraysCore.SVector{3, Int64} to an object of type Tuple{Int64, Int64, Int64}

Closest candidates are:
  convert(::Type{T}, !Matched::T) where T<:Tuple
   @ Base essentials.jl:456
  convert(::Type{T}, !Matched::T) where T
   @ Base Base.jl:84
  convert(::Type{T}, !Matched::Tuple{Vararg{Any, N}}) where {N, T<:Tuple}
   @ Base essentials.jl:457
  ...
